# Proyecto de Visualización — Sesión 1: Datos y Estrategia
**Bootcamp Python PY4 · Módulo 1 · Semana 4**

---

En esta primera sesión vamos a sentar las bases del proyecto.
Al terminar deberías tener:

- Los datos cargados directamente desde GitHub
- Ambos archivos combinados en un solo DataFrame limpio
- Una exploración descriptiva completa
- Una estrategia de visualización definida con apoyo de IA

**Dataset:** Biodiversidad en Parques Nacionales de EE.UU.  
**Fuente:** [github Biodiversity - National Parks](https://github.com/Camille-Le-Roy/data_analytics_resources)

Trabajamos con dos archivos:
| Archivo | Descripción |
|---|---|
| `species_info.csv` | 5,824 especies con categoría taxonómica y estado de conservación |
| `observations.csv` | 23,296 registros de avistamientos por especie y parque |


## 1. Setup: librerías y configuración

Antes de cargar datos, importamos todo lo que vamos a usar durante las tres sesiones.
Si trabajas en Google Colab, `plotly` y `altair` ya vienen instalados.


In [ ]:
# Instalación (descomenta si es necesario)
!pip install altair plotly seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo global
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Librerías cargadas.")


Librerías cargadas.


## 2. Carga de datos desde GitHub

Cargamos los dos archivos directamente desde el repositorio usando sus URLs raw.
No hace falta descargar nada — `pd.read_csv()` acepta URLs.


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/Camille-Le-Roy/data_analytics_resources/main/"

species = pd.read_csv(BASE_URL + "species_info.csv")
observations = pd.read_csv(BASE_URL + "observations.csv")

print(f"species_info:  {species.shape[0]:,} filas × {species.shape[1]} columnas")
print(f"observations:  {observations.shape[0]:,} filas × {observations.shape[1]} columnas")


species_info:  5,824 filas × 4 columnas
observations:  16,378 filas × 3 columnas


### Primer vistazo a cada archivo

Antes de combinarlos, exploramos cada uno por separado para entender su estructura.


In [ ]:
# Vista de species_info
species.head(8)


,category,scientific_name,common_names,conservation_status
0,Mammal,Clethrionomys gapperi gapperi,Gapper's Red-Backed Vole,In Recovery
1,Mammal,Bos bison,"American Bison, Bison",NaN
2,Mammal,Bos taurus,"Aurochs, Aurochs, Domestic Cattle (Feral), Dom...",Species of Concern
3,Mammal,Ovis aries,"Domestic Sheep, Mouflon, Red Sheep, Sheep (Feral)",Species of Concern
4,Mammal,Cervus elaphus,Wapiti Or Elk,NaN
5,Mammal,Odocoileus virginianus,White-Tailed Deer,Threatened
6,Mammal,Sus scrofa,"Feral Hog, Wild Pig",In Recovery
7,Mammal,Canis latrans,Coyote,Species of Concern


In [ ]:
# Vista de observations
observations.head(8)


,scientific_name,park_name,observations
0,Vicia benghalensis,Great Smoky Mountains National Park,68
1,Neovison vison,Great Smoky Mountains National Park,77
2,Githopsis specularioides,Great Smoky Mountains National Park,85
3,Elymus multisetus,Great Smoky Mountains National Park,39
4,Fabronia ciliaris,Great Smoky Mountains National Park,72
5,Lunaria annua,Great Smoky Mountains National Park,65
6,Scirpus cyperinus,Great Smoky Mountains National Park,73
7,Festuca filiformis,Great Smoky Mountains National Park,79


## 3. Exploración inicial

Respondemos las preguntas básicas antes de tocar cualquier dato:
- ¿Cuántos valores nulos hay y dónde?
- ¿Cuáles son los valores únicos de las variables clave?
- ¿Qué forma tiene la distribución de las variables principales?


In [ ]:
# Valores nulos en species_info
print("=== species_info: nulos ===")
print(species.isnull().sum())
print()

# Valores únicos en columnas categóricas clave
print("Categorías taxonómicas:", species["category"].unique())
print()
print("Estados de conservación:", species["conservation_status"].unique())


category                  0
scientific_name           0
common_names              0
conservation_status    2959
dtype: int64


In [ ]:
# Valores nulos en observations
print("=== observations: nulos ===")
print(observations.isnull().sum())
print()

# Parques únicos
print("Parques nacionales:", observations["park_name"].unique())


=== observations: nulos ===
scientific_name    0
park_name          0
observations       0
dtype: int64

Parques nacionales: ['Great Smoky Mountains National Park' 'Yellowstone National Park'
 'Bryce National Park' 'Yosemite National Park']


In [ ]:
# Distribución del estado de conservación
# Nota: muchas especies NO tiene estado asignado (campo vacío = sin amenaza registrada)
print("Distribución de conservation_status:")
print(species["conservation_status"].value_counts(dropna=False))


conservation_status
NaN                   2959
Species of Concern    1681
Endangered             481
Threatened             409
In Recovery            294
Name: count, dtype: int64


## 4. Limpieza de datos

Dos decisiones de limpieza importantes:

1. **`conservation_status` vacío** no significa dato faltante — significa que la especie
   no tiene una categoría de amenaza asignada. La rellenamos con `"No Concern"` para
   que sea explícita en los gráficos.

2. **Nombres de parques** son largos — creamos una versión corta para las visualizaciones.


In [ ]:
# Rellenamos el estado vacío con una etiqueta explícita (por ejemplo, conservation status NaN --> "No Conern")
species["conservation_status"] = species["conservation_status"].fillna("No Concern")

# Verificamos el resultado
print(species["conservation_status"].value_counts())


conservation_status
No Concern            2959
Species of Concern    1681
Endangered             481
Threatened             409
In Recovery            294
Name: count, dtype: int64


In [ ]:
# Nombres cortos de parques para los gráficos (para simplificar el analysis)
park_names = {
    "Great Smoky Mountains National Park": "Great Smoky Mts.",
    "Yellowstone National Park": "Yellowstone",
    "Yosemite National Park": "Yosemite",
    "Bryce National Park": "Bryce Canyon"
}

observations["park_short"] = observations["park_name"].map(park_names)
# El método .map() busca cada valor de la columna "park_name" dentro de las claves del diccionario park_names y lo reemplaza por su versión abreviada

observations.head(4)


,park_name
0,Great Smoky Mts.
1,Great Smoky Mts.
2,Great Smoky Mts.
3,Great Smoky Mts.
4,Great Smoky Mts.
...,...
16373,Yosemite
16374,Yosemite
16375,Yosemite
16376,Yosemite


## 5. Combinación de los dos archivos (merge)

Unimos `species_info` y `observations` usando `scientific_name` como llave común.
Usamos un `left join` desde `observations` para conservar todos los avistamientos,
incluso si una especie no tiene información taxonómica completa.


In [ ]:
# creamos un nuevo dataset que combina 'species_info' y 'observations'
df = observations.merge(species, on="scientific_name", how="left")

print(f"DataFrame combinado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()


DataFrame combinado: 18,052 filas × 7 columnas


,scientific_name,park_name,observations,park_short,category,common_names,conservation_status
0,Vicia benghalensis,Great Smoky Mountains National Park,68,Great Smoky Mts.,Vascular Plant,"Purple Vetch, Reddish Tufted Vetch",Species of Concern
1,Neovison vison,Great Smoky Mountains National Park,77,Great Smoky Mts.,Mammal,American Mink,No Concern
2,Githopsis specularioides,Great Smoky Mountains National Park,85,Great Smoky Mts.,Vascular Plant,Common Bluecup,No Concern
3,Elymus multisetus,Great Smoky Mountains National Park,39,Great Smoky Mts.,Vascular Plant,"Big Squirreltail, Big Wild Rye",Threatened
4,Fabronia ciliaris,Great Smoky Mountains National Park,72,Great Smoky Mts.,Nonvascular Plant,Fabronia Moss,No Concern


Lo que hace el **left join**:

*   **Conserva todo lo de la izquierda:** Toma el dataframe `observations` como base y mantiene todas sus filas intactas.
*   **Busca coincidencias a la derecha:** Va al dataframe `species` y busca la información que coincida exactamente en la columna `"scientific_name"`.
*   **Agrega los datos:**
    *   Si encuentra el nombre científico, añade las columnas de `species` (como el nombre común o la categoría) a esa fila en el nuevo dataframe `df`.
    *   Si no lo encuentra, rellena esos espacios con `NaN` (valores nulos).


In [ ]:
# Revisamos nulos después del merge
print("Nulos tras el merge:")
print(df.isnull().sum())


Nulos tras el merge:
scientific_name        0
park_name              0
observations           0
park_short             0
category               0
common_names           0
conservation_status    0
dtype: int64


In [ ]:
# Limpiamos nulos residuales del merge
df["conservation_status"] = df["conservation_status"].fillna("No Concern")
df["category"] = df["category"].fillna("Unknown")

print("Dataset final listo:", df.shape)


Dataset final listo: (18052, 7)


## 6. Estadísticas descriptivas

Antes de visualizar, calculamos algunos números clave que guiarán
nuestras decisiones de diseño más adelante.


In [ ]:
# Avistamientos totales por parque
print("Total de observaciones por parque:")
print(df.groupby("park_name")["observations"].sum().sort_values(ascending=False))


Total de observaciones por parque:
park_name
Yellowstone National Park              1625402
Yosemite National Park                  999867
Bryce National Park                     635348
Great Smoky Mountains National Park     477339
Name: observations, dtype: int64


In [ ]:
# Especies únicas por categoría taxonómica
print("Especies únicas por categoría:")
print(species.groupby("category")["scientific_name"].nunique().sort_values(ascending=False))


Especies únicas por categoría:
category
Vascular Plant       4262
Bird                  488
Nonvascular Plant     333
Mammal                176
Fish                  125
Amphibian              79
Reptile                78
Name: scientific_name, dtype: int64


In [ ]:
# Especies amenazadas: cuántas hay por categoría taxonómica
amenazadas = species[species["conservation_status"] != "No Concern"]

print(f"Especies con algún estado de amenaza: {len(amenazadas)}")
print()
print(amenazadas.groupby(["category", "conservation_status"])
      .size()
      .unstack(fill_value=0))


Especies con algún estado de amenaza: 2865

conservation_status  Endangered  In Recovery  Species of Concern  Threatened
category                                                                    
Amphibian                    15            6                  24          10
Bird                         51           22                 182          25
Fish                         18            3                  39          20
Mammal                       45           15                  72          26
Nonvascular Plant            14           11                  85          14
Reptile                       9            9                  23           4
Vascular Plant              329          228                1256         310


## 7. Estrategia con IA (Prompt Engineering)

Antes de programar gráficos, consultamos con una IA para definir
la mejor estrategia visual para este dataset.

**Tu tarea:**
1. Escribe un prompt describiendo el dataset y el objetivo del dashboard
2. Ejecuta el prompt en ChatGPT, Claude u otra herramienta
3. Completa las celdas de abajo con el prompt usado y la estrategia elegida

**Prompt base sugerido** (adáptalo y mejóralo):

> *"Tengo un dataset de biodiversidad en parques nacionales de EE.UU.
> con dos tablas: una con 5,824 especies (categoría taxonómica, nombre científico,
> estado de conservación: Endangered/Threatened/Species of Concern/In Recovery)
> y otra con 23,296 registros de avistamientos por especie y parque (4 parques).
> Quiero construir un dashboard EDA en Python usando Seaborn, Plotly y Altair.
> ¿Qué 5 visualizaciones me recomendarías para contar una historia sobre
> el estado de amenaza de la biodiversidad en estos parques?
> Justifica cada gráfico con una pregunta de negocio concreta."*


**Algunos consejos para un buen Prompt Engineering**

- Sé específico y evita ambigüedades
- Proporciona contexto (muestra la estructura de tu dataset)
- Define la audiencia
- Establece restricciones claras
- Itera y refina progresivamente
- Experimenta y ajusta el prompt


### Prompt utilizado

> Prompt base sugerido

---

### Respuesta de la IA (resumen)

# Visualizaciones sugeridas

## Viz 01 — Barras apiladas al 100% (Matplolib)

**Pregunta:** ¿Qué taxones concentran una mayor presión relativa?

El porcentaje normalizado al 100% elimina el sesgo asociado al tamaño de cada grupo. No es equivalente que el 30% de los peces esté amenazado a que el 30% de los insectos lo esté. Utilizando `hue_order` y una paleta tipo semáforo (rojo → naranja → amarillo → verde), es posible identificar rápidamente qué grupos presentan una mayor proporción de especies en riesgo.

**Valor:** una visión general del estado de conservación de los distintos taxones en una sola figura.

---

## Viz 02 — Heatmap (Seaborn)

**Pregunta:** ¿Qué parques actúan como refugios importantes para cada taxón?

Una matriz de calor que cruza parques y taxones permite identificar patrones geográficos difíciles de detectar en una tabla. Un parque con muchas especies amenazadas de reptiles pero pocas de mamíferos puede requerir estrategias de conservación diferentes.

El uso de `annot=True` permite mostrar los conteos exactos sin perder la visión global de los datos.

**Valor:** identificación rápida de prioridades de conservación por parque y grupo taxonómico.

---

## Viz 03 — Treemap interactivo (Plotly)

**Pregunta:** ¿Cuál es el volumen real de especies en cada categoría de riesgo?

Mientras que las barras muestran proporciones, el treemap permite visualizar cantidades absolutas. La estructura jerárquica (categoría de conservación → taxón) facilita explorar los datos mediante interacción y etiquetas emergentes (*hover*).

**Valor:** comprender la magnitud real de cada categoría de conservación y de cada grupo taxonómico.

---

## Viz 04 — Bubble Plot por parque (Plotly)

**Pregunta:** ¿Existe relación entre el esfuerzo de observación y la carga de conservación?

Esta visualización combina simultáneamente:

- Parque
- Número de observaciones
- Número de especies amenazadas

El tamaño de las burbujas representa la cantidad de especies en riesgo. Un parque con pocas observaciones pero muchas especies amenazadas puede señalar una necesidad de monitoreo adicional.

**Valor:** detectar áreas donde la conservación podría estar insuficientemente documentada.

---

## Viz 05 — Strip Plot + Box Plot (Altair)

**Pregunta:** ¿Dónde están ocurriendo las recuperaciones más exitosas?

Al filtrar únicamente las especies clasificadas como *In Recovery*, esta visualización muestra la distribución de observaciones por parque.

La combinación de:

- `mark_point()` (datos individuales)
- `mark_boxplot()` (resumen estadístico)

permite observar tanto los valores individuales como las tendencias generales.

**Valor:** identificar los parques donde los programas de recuperación parecen estar obteniendo mejores resultados.

---


## 8. Checkpoint: guardar el DataFrame limpio

Guardamos el DataFrame combinado para no repetir los pasos de carga
y limpieza en las sesiones 2 y 3.


In [ ]:
df.to_csv("biodiversity_clean.csv", index=False)
print("Archivo guardado: biodiversity_clean.csv")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.dtypes


---
## Checklist de cierre — Sesión 1

Antes de terminar, verifica que completaste lo siguiente:

- [ ] Datos cargados correctamente desde GitHub (sin errores)
- [ ] `species_info` y `observations` explorados por separado
- [ ] `conservation_status` vacío reemplazado por `"No Concern"`
- [ ] Merge realizado correctamente (23,296 filas conservadas)
- [ ] Estadísticas descriptivas calculadas (por parque, por categoría, por amenaza)
- [ ] Prompt de IA ejecutado y estrategia de visualización definida
- [ ] `biodiversity_clean.csv` guardado

**Próxima sesión:** construimos las visualizaciones estáticas con Seaborn
e interactivas con Plotly.
